# 1. Load all tables
## check cell loads most tables but not alerts

In [1]:
from pathlib import Path
import pandas as pd

DATA = Path(".")

stop_visits = pd.read_parquet(DATA / "stop_visits.parquet")
vehicles = pd.read_parquet(DATA / "vehicle_positions.parquet")
alerts = pd.read_parquet(DATA / "service_alerts.parquet")
routes = pd.read_parquet(DATA / "gtfs_routes.parquet")
stops = pd.read_parquet(DATA / "gtfs_stops.parquet")

# 2.Settings and functions
## Defines all the functions

In [2]:
TZ = "Europe/Oslo"
STALE_S = 300                      # vehicle position older than 5 min = stale
MAX_ABS_DELAY_S = 3 * 3600         # delays over 3 h are treated as errors
ON_TIME_EARLY_S, ON_TIME_LATE_S = -60, 180   # on time = 1 min early to 3 min late


def to_utc(series):
    return pd.to_datetime(series, utc=True, format="ISO8601")


def flag(series):
    return series.fillna(False).astype(bool)


def build_dims(routes, stops):
    dim_routes = routes.rename(columns={"route_id": "line_ref", "route_short_name": "public_code",
                                        "route_long_name": "line_name"})[
        ["line_ref", "public_code", "line_name", "route_type"]]
    dim_stops = stops.rename(columns={"stop_id": "stop_ref", "stop_name": "gtfs_stop_name"})[
        ["stop_ref", "gtfs_stop_name", "stop_lat", "stop_lon"]].copy()
    dim_stops["stop_lat"] = pd.to_numeric(dim_stops["stop_lat"], errors="coerce")
    dim_stops["stop_lon"] = pd.to_numeric(dim_stops["stop_lon"], errors="coerce")
    return dim_routes, dim_stops


def silver_stop_visits(stop_visits, dim_routes, dim_stops):
    # 1. final state: latest snapshot of each stop visit
    sv = (stop_visits.sort_values("snapshot_time")
          .drop_duplicates(["data_frame_ref", "journey_ref", "stop_order"], keep="last")
          .copy())
    # 2. all timestamps to UTC
    for c in ["aimed_arrival", "expected_arrival", "actual_arrival",
              "aimed_departure", "expected_departure", "actual_departure"]:
        sv[c] = to_utc(sv[c])

    # 3. delay = observed - planned (departure if possible, otherwise arrival)
    sv["is_actual"] = sv["call_type"].eq("recorded")
    obs_dep = sv["actual_departure"].where(sv["is_actual"], sv["expected_departure"])
    obs_arr = sv["actual_arrival"].where(sv["is_actual"], sv["expected_arrival"])
    sv["aimed_time"] = sv["aimed_departure"].fillna(sv["aimed_arrival"])
    sv["observed_time"] = obs_dep.fillna(obs_arr)
    dep_delay = (obs_dep - sv["aimed_departure"]).dt.total_seconds()
    arr_delay = (obs_arr - sv["aimed_arrival"]).dt.total_seconds()
    sv["delay_s"] = dep_delay.fillna(arr_delay)

    # 4. flags
    sv["cancelled"] = flag(sv["call_cancelled"]) | flag(sv["journey_cancelled"])
    sv["no_prediction"] = ~sv["is_actual"] & sv["delay_s"].eq(0)
    sv["delay_outlier"] = sv["delay_s"].abs() > MAX_ABS_DELAY_S
    sv["on_time"] = sv["delay_s"].between(ON_TIME_EARLY_S, ON_TIME_LATE_S)

    # 5. local time for analysis (planned time, Norway)
    local = sv["aimed_time"].dt.tz_convert(TZ)
    sv["service_date"] = local.dt.date.astype("string")
    sv["hour"] = local.dt.hour
    sv["weekday"] = local.dt.day_name()

    # 6. line numbers and stop coordinates from GTFS
    sv = sv.merge(dim_routes, on="line_ref", how="left").merge(dim_stops, on="stop_ref", how="left")
    sv["stop_name"] = sv["stop_name"].fillna(sv["gtfs_stop_name"])

    cols = ["data_frame_ref", "journey_ref", "line_ref", "public_code", "line_name", "vehicle_mode",
            "direction_ref", "operator_ref", "stop_order", "stop_ref", "stop_name", "stop_lat", "stop_lon",
            "destination", "aimed_time", "observed_time", "delay_s", "is_actual", "on_time",
            "cancelled", "no_prediction", "delay_outlier", "service_date", "hour", "weekday",
            "aimed_arrival", "aimed_departure", "actual_arrival", "actual_departure",
            "expected_arrival", "expected_departure", "snapshot_time", "source_file"]
    return sv[cols].sort_values(["journey_ref", "stop_order"]).reset_index(drop=True)


def silver_vehicle_positions(vehicles, line_modes):
    vp = (vehicles.sort_values("snapshot_time")
          .drop_duplicates(["vehicle_id", "last_updated"], keep="first")   # same position seen twice
          .copy())
    vp["last_updated"] = to_utc(vp["last_updated"])
    vp["age_s"] = (vp["snapshot_time"] - vp["last_updated"]).dt.total_seconds()
    vp["is_stale"] = vp["age_s"] > STALE_S
    vp["bearing"] = vp["bearing"] % 360
    vp = vp.merge(line_modes, on="line_ref", how="left")
    vp = vp.dropna(subset=["latitude", "longitude"])
    local = vp["last_updated"].dt.tz_convert(TZ)
    vp["hour"] = local.dt.hour
    vp["weekday"] = local.dt.day_name()
    return vp.drop(columns=["speed"]).reset_index(drop=True)          # speed is always empty


CATEGORIES = [("trafikk", "traffic"), ("forrige rute", "knock-on"), ("teknisk", "technical"),
              ("vær", "weather"), ("korresponder", "waiting for connection"), ("innstil", "cancelled")]


def categorise(summary):
    text = (summary or "").lower()
    for word, category in CATEGORIES:
        if word in text:
            return category
    return "other"


def silver_alerts(alerts):
    keys = ["situation_number", "version_time", "affect_type", "line_ref", "journey_ref"]
    seen = (alerts.groupby(keys, dropna=False)["snapshot_time"]
            .agg(first_seen="min", last_seen="max").reset_index())
    al = (alerts.sort_values("snapshot_time").drop_duplicates(keys, keep="last")
          .drop(columns=["snapshot_time"]).merge(seen, on=keys, how="left"))
    for c in ["creation_time", "version_time", "valid_start", "valid_end", "origin_aimed_departure"]:
        al[c] = to_utc(al[c])
    al["category"] = al["summary"].map(categorise)
    return al.reset_index(drop=True)

# 3. Build stop visits

In [3]:
dim_routes, dim_stops = build_dims(routes, stops)
sv = silver_stop_visits(stop_visits, dim_routes, dim_stops)

print(f"{len(stop_visits):,} bronze rows -> {len(sv):,} unique stop visits")
print(sv["is_actual"].value_counts())
sv.head()

1,688,126 bronze rows -> 157,480 unique stop visits
is_actual
True     154039
False      3441
Name: count, dtype: int64


,data_frame_ref,journey_ref,line_ref,public_code,line_name,vehicle_mode,direction_ref,operator_ref,stop_order,stop_ref,...,hour,weekday,aimed_arrival,aimed_departure,actual_arrival,actual_departure,expected_arrival,expected_departure,snapshot_time,source_file
0,2026-10-30,KOL:ServiceJourney:1001_251008118343317_1001,KOL:Line:8_1001,1,,bus,0,KOL:Operator:309,1,NSR:Quay:46951,...,5,Friday,NaT,2026-10-30 04:47:00+00:00,NaT,2026-10-30 04:48:19.316000+00:00,NaT,NaT,2026-10-30 05:31:53+00:00,estimated_timetable/date=2026-10-30/estimated_...
1,2026-10-30,KOL:ServiceJourney:1001_251008118343317_1001,KOL:Line:8_1001,1,,bus,0,KOL:Operator:309,2,NSR:Quay:47003,...,5,Friday,2026-10-30 04:48:00+00:00,2026-10-30 04:48:00+00:00,2026-10-30 04:49:43.313000+00:00,2026-10-30 04:49:43.313000+00:00,NaT,NaT,2026-10-30 05:31:53+00:00,estimated_timetable/date=2026-10-30/estimated_...
2,2026-10-30,KOL:ServiceJourney:1001_251008118343317_1001,KOL:Line:8_1001,1,,bus,0,KOL:Operator:309,3,NSR:Quay:46920,...,5,Friday,2026-10-30 04:49:00+00:00,2026-10-30 04:49:00+00:00,2026-10-30 04:50:17.312000+00:00,2026-10-30 04:50:17.312000+00:00,NaT,NaT,2026-10-30 05:31:53+00:00,estimated_timetable/date=2026-10-30/estimated_...
3,2026-10-30,KOL:ServiceJourney:1001_251008118343317_1001,KOL:Line:8_1001,1,,bus,0,KOL:Operator:309,4,NSR:Quay:46033,...,5,Friday,2026-10-30 04:50:00+00:00,2026-10-30 04:50:00+00:00,2026-10-30 04:51:02.315000+00:00,2026-10-30 04:51:02.315000+00:00,NaT,NaT,2026-10-30 05:31:53+00:00,estimated_timetable/date=2026-10-30/estimated_...
4,2026-10-30,KOL:ServiceJourney:1001_251008118343317_1001,KOL:Line:8_1001,1,,bus,0,KOL:Operator:309,5,NSR:Quay:46836,...,5,Friday,2026-10-30 04:51:00+00:00,2026-10-30 04:51:00+00:00,2026-10-30 04:52:08.325000+00:00,2026-10-30 04:52:08.325000+00:00,NaT,NaT,2026-10-30 05:31:53+00:00,estimated_timetable/date=2026-10-30/estimated_...


# 4. Build vehicle positions and alerts

In [4]:
line_modes = sv.groupby("line_ref")["vehicle_mode"].first().reset_index()

vp = silver_vehicle_positions(vehicles, line_modes)
al = silver_alerts(alerts)

print(f"vehicles: {len(vehicles):,} -> {len(vp):,} rows, stale: {vp['is_stale'].mean():.0%}")
print(f"alerts:   {len(alerts):,} -> {len(al):,} rows")
print(al["category"].value_counts())

vehicles: 53,443 -> 43,970 rows, stale: 0%
alerts:   75 -> 38 rows
category
traffic                   13
knock-on                  13
other                      7
technical                  4
waiting for connection     1
Name: count, dtype: int64


# 5. Quality checks on line numbers and coordinates

In [5]:
print("stop visits with a line number: ", f"{sv['public_code'].notna().mean():.1%}")
print("stop visits with coordinates:   ", f"{sv['stop_lat'].notna().mean():.1%}")
print("vehicles with bus/ferry known:  ", f"{vp['vehicle_mode'].notna().mean():.1%}")
print()
actual_buses = sv[sv["is_actual"] & (sv["vehicle_mode"] == "bus")]
print(f"actual bus stop visits: {len(actual_buses):,}")
print(f"  cancelled:     {actual_buses['cancelled'].mean():.1%}")
print(f"  outliers:      {actual_buses['delay_outlier'].mean():.2%}")
print(f"  on time:       {actual_buses['on_time'].mean():.1%}")
print(actual_buses["delay_s"].describe().round(0))

stop visits with a line number:  100.0%
stop visits with coordinates:    100.0%
vehicles with bus/ferry known:   99.2%

actual bus stop visits: 153,180
  cancelled:     0.0%
  outliers:      0.00%
  on time:       55.3%
count    153180.0
mean        194.0
std         346.0
min       -2927.0
25%          40.0
50%         129.0
75%         273.0
max       10217.0
Name: delay_s, dtype: float64


# 6. Save the silver layer tables
## Creating silver folder with silver layer tables

In [6]:
import os
os.makedirs("silver", exist_ok=True)
sv.to_parquet("silver/stop_visits.parquet", index=False)
vp.to_parquet("silver/vehicle_positions.parquet", index=False)
al.to_parquet("silver/alerts.parquet", index=False)
dim_stops.to_parquet("silver/stops.parquet", index=False)
dim_routes.to_parquet("silver/routes.parquet", index=False)
print(os.listdir("silver"))

['alerts.parquet', 'routes.parquet', 'stops.parquet', 'stop_visits.parquet', 'vehicle_positions.parquet']
